In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from mtrain.utils import show, DiskImage, DiskBooleanMask

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from PIL import Image, ImageDraw
import requests

In [ ]:
from mtrain.neg_mask.ipywidgets.widget_7 import ImageReviewWidget
INAT = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/flowers/inat/")

In [ ]:
# Initialize Grounding DINO model
model_id = "IDEA-Research/grounding-dino-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)

In [ ]:
def detect_flowers_grounding_dino(image_path, text_prompt="flower", confidence_threshold=0.3):
    """
    Detect flowers in an image using Grounding DINO.
    
    Args:
        image_path: Path to the image file
        text_prompt: Text prompt to detect (default: "flower")
        confidence_threshold: Minimum confidence for detections
        
    Returns:
        tuple: (image, detections) where detections contains boxes, scores, labels
    """
    # Load image
    image = Image.open(image_path)
    
    # Process inputs
    inputs = processor(images=image, text=text_prompt, return_tensors="pt").to(device)
    
    # Run inference
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Process outputs
    results = processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        threshold=confidence_threshold,
        text_threshold=confidence_threshold,
        target_sizes=[image.size[::-1]]  # (height, width)
    )
    
    return image, results[0]

In [ ]:
def draw_detections(image, detections, show_labels=True, show_scores=True):
    """
    Draw bounding boxes on image with labels and scores.
    
    Args:
        image: PIL Image
        detections: Detection results from Grounding DINO
        show_labels: Whether to show labels
        show_scores: Whether to show confidence scores
        
    Returns:
        PIL Image with drawn bounding boxes
    """
    draw = ImageDraw.Draw(image)
    
    boxes = detections["boxes"]
    scores = detections["scores"] 
    labels = detections["labels"]
    
    for box, score, label in zip(boxes, scores, labels):
        # Convert box format (x_center, y_center, width, height) to (x1, y1, x2, y2)
        x1, y1, x2, y2 = box.tolist()
        
        # Draw rectangle
        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
        
        # Create label text
        label_text = ""
        if show_labels:
            label_text += f"{label}"
        if show_scores:
            if label_text:
                label_text += f" ({score:.2f})"
            else:
                label_text = f"{score:.2f}"
        
        # Draw label background and text
        if label_text:
            # Get text size for background
            bbox = draw.textbbox((x1, y1-20), label_text)
            draw.rectangle(bbox, fill="red")
            draw.text((x1, y1-20), label_text, fill="white")
    
    return image


def get_bounding_boxes(image_path, text_prompt="flower", confidence_threshold=0.3):
    bboxes = []
    image, detections = detect_flowers_grounding_dino(image_path, text_prompt, confidence_threshold)
    
    # Print detection info
    num_detections = len(detections["boxes"])
    print(f"Found {num_detections} detections with confidence > {confidence_threshold}")
    
    if num_detections > 0:
        for i, (box, score, label) in enumerate(zip(detections["boxes"], detections["scores"], detections["labels"])):
            x1, y1, x2, y2 = box.tolist()
            bboxes.append({"x1": x1, "y1": y1, "x2": x2, "y2": y2})
    return bboxes


def visualize_flower_detection(image_path, text_prompt="flower", confidence_threshold=0.3):
    """
    Detect and visualize flowers in an image.
    
    Args:
        image_path: Path to the image file
        text_prompt: Text prompt for detection
        confidence_threshold: Minimum confidence threshold
    """
    # Detect flowers
    image, detections = detect_flowers_grounding_dino(image_path, text_prompt, confidence_threshold)
    
    # Print detection info
    num_detections = len(detections["boxes"])
    print(f"Found {num_detections} detections with confidence > {confidence_threshold}")
    
    if num_detections > 0:
        for i, (box, score, label) in enumerate(zip(detections["boxes"], detections["scores"], detections["labels"])):
            x1, y1, x2, y2 = box.tolist()
            print(f"Detection {i+1}: {label} (score: {score:.3f}) at [{x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f}]")
    
    # Draw detections
    result_image = draw_detections(image.copy(), detections)
    
    # Display using matplotlib
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))
    
    ax1.imshow(image)
    ax1.set_title("Original Image")
    ax1.axis('off')
    
    ax2.imshow(result_image)
    ax2.set_title(f"Flower Detections (n={num_detections})")
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return image, detections, result_image

In [ ]:
# Test with a sample image from your iNaturalist photos
photos_dir = INAT / "photos"
sample_images = list(photos_dir.glob("*.jpg"))[:5]  # Get first 5 images

print(f"Found {len(sample_images)} sample images in {photos_dir}")
for i, img_path in enumerate(sample_images):
    print(f"{i+1}. {img_path.name}")

In [ ]:
# Test flower detection on the first sample image
if sample_images:
    test_image = sample_images[2]
    print(f"Testing flower detection on: {test_image.name}")
    
    # Try different text prompts and confidence thresholds
    text_prompts = ["flower"]
    confidence_threshold = 0.3
    
    for prompt in text_prompts:
        print(f"\n--- Testing with prompt: '{prompt}' ---")
        try:
            image, detections, result_image = visualize_flower_detection(
                test_image, 
                text_prompt=prompt, 
                confidence_threshold=confidence_threshold
            )
            break  # Use the first successful detection
        except Exception as e:
            print(f"Error with prompt '{prompt}': {e}")
else:
    print("No sample images found!")

In [ ]:
from tqdm import tqdm
from mtrain.utils import mkdir
import json

OUT_DIR = mkdir(INAT / "bboxes")
photos_dir = INAT / "photos"
images = list(photos_dir.glob("*.jpg"))
for path in tqdm(images):
    dest = OUT_DIR / f"{path.stem}.json"
    if dest.exists():
        continue
    bboxes = get_bounding_boxes(path)
    with open(dest, "w") as f:
        json.dump({"bboxes": bboxes}, f)

In [ ]:
widget = ImageReviewWidget(INAT / "photos", INAT / "selected", INAT / "skipped")

In [ ]:
widget.ui()